# 03 — Shared scene crops and reconstruction inputs

This CPU notebook makes the vehicle larger without modifying the eight-view geometry independently. For each car it computes **one union crop from its eight selected training masks**, adds a configurable margin, and applies that same normalized rectangle to every view. Original BiRefNet files remain unchanged.

It exports three reproducible derivatives: high-resolution cropped inputs for 3DGS, one shared 512-based set for DUSt3R/MASt3R, and a 14-pixel-grid set for VGGT. Small inputs are never enlarged. The manifest records crop, scale, and padding values for later camera-intrinsic transformation.


In [ ]:
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
import torch

CPU_ONLY_NOTEBOOK = True
GPU_ATTACHED = torch.cuda.is_available()

print("Runtime check")
print("-" * 50)
print("GPU attached:", GPU_ATTACHED)

if GPU_ATTACHED:
    print("GPU:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_bytes / 1024**3:.2f} GB free / {total_bytes / 1024**3:.2f} GB total")

if CPU_ONLY_NOTEBOOK and GPU_ATTACHED:
    raise RuntimeError(
        "This notebook is CPU-only. To conserve Colab GPU availability, "
        "change the Colab hardware accelerator to None, reconnect the CPU "
        "kernel in VS Code, and run the notebook again."
    )

print("Correct CPU runtime: continue with the notebook.")


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))

# Change only this value if the Drive project is moved.
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"

def find_unique_dir(names, search_roots):
    direct = [root / name for root in search_roots for name in names]
    matches = [p for p in direct if p.is_dir()]
    if not matches:
        for root in search_roots:
            if root.is_dir():
                matches.extend(p for p in root.rglob("*") if p.is_dir() and p.name.casefold() in {n.casefold() for n in names})
    unique = list(dict.fromkeys(p.resolve() for p in matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected one of {names}; found {len(unique)}: {unique}")
    return unique[0]

SEARCH_ROOTS = [PROJECT_ROOT / "data", PROJECT_ROOT]
INDUSTRIAL_ROOT = find_unique_dir(["IndustrialInventory"], SEARCH_ROOTS)
HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], SEARCH_ROOTS)

# If HQ200 is a wrapper folder, descend to the folder containing capture scenes.
if (HQ200_ROOT / "3DrealCarHQ200").is_dir():
    HQ200_ROOT = HQ200_ROOT / "3DrealCarHQ200"

print("Project:   ", PROJECT_ROOT)
print("Industrial:", INDUSTRIAL_ROOT)
print("3DRealCar: ", HQ200_ROOT)


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"

if not (CODE_ROOT / "code" / "src").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

CODE_PACKAGE_ROOT = CODE_ROOT / "code"
if str(CODE_PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_PACKAGE_ROOT))

print("Reusable code:", CODE_ROOT / "code" / "src")


In [ ]:
from src.image_preprocessing import (
    VIEW_LABELS, contain_without_upscale, method_canvas,
    pixel_crop_box, shared_normalized_crop,
)

BIREFNET_ROOT = PROJECT_ROOT / "data_processed" / "birefnet_original_resolution"
BIREFNET_MANIFEST = BIREFNET_ROOT / "manifest.csv"
SELECTION_PATH = PROJECT_ROOT / "splits" / "sparse8" / "hq200_manual_8views.csv"
OUTPUT_ROOT = PROJECT_ROOT / "data_processed" / "method_inputs"
RUN_EXPORT = False
OVERWRITE = False
CROP_MARGIN = 0.10
EXCLUDED_SCENES = {"car_12104835"}

if not BIREFNET_MANIFEST.is_file():
    raise FileNotFoundError(f"Run notebook 01 in full mode first: {BIREFNET_MANIFEST}")
if not SELECTION_PATH.is_file():
    raise FileNotFoundError(f"Finish and save notebook 02 annotations first: {SELECTION_PATH}")


In [ ]:
manifest = pd.read_csv(BIREFNET_MANIFEST)
manifest = manifest[manifest.status.isin(["written", "skipped"])].copy()
manifest = manifest[manifest.output_image.notna() & manifest.output_mask.notna()].copy()
manifest = manifest[~manifest.scene.astype(str).str.casefold().isin({x.casefold() for x in EXCLUDED_SCENES})]
manual = pd.read_csv(SELECTION_PATH)

rows = []
for (dataset, scene), group in manifest.groupby(["dataset", "scene"], sort=True):
    group = group.copy()
    if dataset == "3DRealCar":
        selected = manual[manual.scene.astype(str).eq(str(scene))].copy()
        if len(selected) != 8:
            raise ValueError(f"{scene}: expected 8 saved HQ annotations, found {len(selected)}")
        order_map = dict(zip(selected.source.astype(str), selected.view_order.astype(int)))
        group["split"] = np.where(group.source.astype(str).isin(order_map), "train", "test")
        group["view_order"] = group.source.astype(str).map(order_map)
    else:
        if len(group) != 8:
            print(f"Skipping {scene}: Industrial scene has {len(group)} rather than 8 images")
            continue
        group["split"] = "train"
        group["view_order"] = range(8)
    rows.append(group)

inputs = pd.concat(rows, ignore_index=True)
train_counts = inputs.query("split == 'train'").groupby(["dataset", "scene"]).size()
assert train_counts.eq(8).all(), "Every retained scene must have exactly eight reconstruction inputs."
display(train_counts.rename("training_views").to_frame())


In [ ]:
crop_rows = []
for (dataset, scene), group in inputs.groupby(["dataset", "scene"], sort=True):
    training = group[group.split.eq("train")].sort_values("view_order")
    box = shared_normalized_crop(training.output_mask, margin_fraction=CROP_MARGIN)
    crop_sizes = []
    for row in group.itertuples(index=False):
        with Image.open(row.output_image) as image:
            px = pixel_crop_box(box, image.size)
        crop_sizes.append((px[2] - px[0], px[3] - px[1]))
    crop_rows.append({
        "dataset": dataset, "scene": scene,
        "crop_x0_norm": box[0], "crop_y0_norm": box[1],
        "crop_x1_norm": box[2], "crop_y1_norm": box[3],
        "scene_width": min(x[0] for x in crop_sizes),
        "scene_height": min(x[1] for x in crop_sizes),
    })
scene_crops = pd.DataFrame(crop_rows)
display(scene_crops)


In [ ]:
def export_row(row, crop):
    box_norm = tuple(crop[x] for x in ["crop_x0_norm", "crop_y0_norm", "crop_x1_norm", "crop_y1_norm"])
    with Image.open(row.output_image) as opened:
        rgb = opened.convert("RGB")
        crop_box = pixel_crop_box(box_norm, rgb.size)
        cropped_rgb = rgb.crop(crop_box)
    with Image.open(row.output_mask) as opened:
        cropped_mask = opened.convert("L").crop(crop_box)

    scene_size = (int(crop.scene_width), int(crop.scene_height))
    high_rgb, high_scale, high_pad_x, high_pad_y = contain_without_upscale(cropped_rgb, scene_size, (255, 255, 255))
    high_mask, _, _, _ = contain_without_upscale(cropped_mask, scene_size, 0)
    outputs = [("3dgs", high_rgb, high_mask, high_scale, high_pad_x, high_pad_y)]
    for method in ["dust3r_mast3r", "vggt"]:
        canvas = method_canvas(scene_size, method)
        out_rgb, second_scale, second_pad_x, second_pad_y = contain_without_upscale(high_rgb, canvas, (255, 255, 255))
        out_mask, _, _, _ = contain_without_upscale(high_mask, canvas, 0)
        total_scale = second_scale * high_scale
        total_pad_x = second_pad_x + second_scale * high_pad_x
        total_pad_y = second_pad_y + second_scale * high_pad_y
        outputs.append((method, out_rgb, out_mask, total_scale, total_pad_x, total_pad_y))

    records = []
    name = Path(row.output_image).name
    for method, out_rgb, out_mask, scale, pad_x, pad_y in outputs:
        base = OUTPUT_ROOT / method / row.dataset / row.scene
        image_path, mask_path = base / "images" / name, base / "masks" / name
        if OVERWRITE or not (image_path.is_file() and mask_path.is_file()):
            image_path.parent.mkdir(parents=True, exist_ok=True)
            mask_path.parent.mkdir(parents=True, exist_ok=True)
            out_rgb.save(image_path, compress_level=3)
            out_mask.save(mask_path, compress_level=3)
        records.append({
            **row._asdict(), "method": method,
            "crop_x0": crop_box[0], "crop_y0": crop_box[1],
            "crop_x1": crop_box[2], "crop_y1": crop_box[3],
            "scale_after_crop": scale, "pad_x": pad_x, "pad_y": pad_y,
            "method_width": out_rgb.width, "method_height": out_rgb.height,
            "method_image": str(image_path), "method_mask": str(mask_path),
        })
    return records

if RUN_EXPORT:
    records = []
    crop_lookup = scene_crops.set_index(["dataset", "scene"])
    for row in tqdm(inputs.itertuples(index=False), total=len(inputs), desc="Exporting method inputs"):
        records.extend(export_row(row, crop_lookup.loc[(row.dataset, row.scene)]))
    method_manifest = pd.DataFrame(records)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    method_manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
    scene_crops.to_csv(OUTPUT_ROOT / "scene_crops.csv", index=False)
    print("Saved:", OUTPUT_ROOT / "manifest.csv")
else:
    print("Dry run complete. Set RUN_EXPORT=True to create method inputs (GPU is not needed).")


## Intrinsics after preprocessing

If original intrinsics are $(f_x,f_y,c_x,c_y)$, use the recorded crop origin, uniform scale, and padding:

$f'_x=s f_x$, $f'_y=s f_y$, $c'_x=s(c_x-x_0)+p_x$, and $c'_y=s(c_y-y_0)+p_y$.

No new detail is invented: the 3DGS export never enlarges pixels, while the model-specific sets may only downsample. The shared per-scene crop keeps all eight views geometrically consistent.


In [ ]:
def show_all_3dgs_scenes(method_manifest):
    shown = method_manifest[(method_manifest.method == "3dgs") & (method_manifest.split == "train")].copy()
    for dataset in ["3DRealCar", "IndustrialInventory"]:
        part = shown[shown.dataset == dataset]
        scenes = sorted(part.scene.unique())
        fig, axes = plt.subplots(len(scenes), 8, figsize=(24, 3 * len(scenes)), squeeze=False)
        for r, scene in enumerate(scenes):
            group = part[part.scene == scene].sort_values(["view_order", "source"])
            for c, (_, item) in enumerate(group.head(8).iterrows()):
                with Image.open(item.method_image) as image:
                    axes[r, c].imshow(image.convert("RGB"))
                axes[r, c].axis("off")
                if r == 0:
                    axes[r, c].set_title(VIEW_LABELS[c])
            axes[r, 0].set_ylabel(scene, rotation=0, ha="right", labelpad=90, fontsize=8)
        fig.suptitle(f"{dataset}: shared-crop 3DGS inputs")
        plt.tight_layout()
        plt.show()

if RUN_EXPORT:
    show_all_3dgs_scenes(method_manifest)
